# ARIMA 7-Day Rebalance Test

Uses the best ARIMA orders from `ARIMA results/arima_selected_orders.csv` (produced by `02_arima_order_selection.ipynb`)
to generate walk-forward 7-day return forecasts on the test set for each coin.

**Run `02_arima_order_selection.ipynb` first.**

Outputs saved to `ARIMA results/`:
- `arima_7d_rebalance_test_summary.csv` — per-coin test metrics
- `ARIMA_test_7d_rebalance_forecast_matrix.npy/.csv` — forecast return matrix (used by CMVO)
- `ARIMA_test_7d_rebalance_actual_matrix.npy/.csv` — actual return matrix

| Setting | Value |
|---------|-------|
| Rebalance interval | 7 days |
| Walk-forward method | Strict (full refit at each rebalance step) |
| Parallelism | All 8 coins processed in parallel (`joblib`, `n_jobs=-1`) |

In [ ]:
import os
import ast
import pandas as pd
import numpy as np
import warnings
import time
from joblib import Parallel, delayed

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.simplefilter("ignore", ConvergenceWarning)
warnings.simplefilter("ignore", FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Paths and settings

In [ ]:
data_dir      = "klines csv data/prices_cleaned"
selection_dir = "ARIMA results"
output_dir    = "ARIMA results"
os.makedirs(output_dir, exist_ok=True)

selected_orders_path = os.path.join(selection_dir, "arima_selected_orders.csv")

BASE_DATE      = pd.Timestamp("2022-04-01")
REBAL_INTERVAL = 7
TRAIN_RATIO    = 0.6
VAL_RATIO      = 0.2
TEST_RATIO     = 0.2
N_JOBS         = -1   # -1 = use all CPU cores

## Helper functions

In [ ]:
def parse_mixed_time(series, base_date):
    """Handle mixed Binance timestamp formats (relative seconds, Unix seconds, Unix milliseconds)."""
    s = pd.to_numeric(series, errors="coerce")
    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    mask_rel = s.notna() & (s < 10**9)
    mask_sec = s.notna() & (s >= 10**9) & (s < 10**12)
    mask_ms  = s.notna() & (s >= 10**12)
    if mask_rel.any():
        parsed.loc[mask_rel] = base_date + pd.to_timedelta(s.loc[mask_rel], unit="s")
    if mask_sec.any():
        parsed.loc[mask_sec] = pd.to_datetime(s.loc[mask_sec], unit="s", errors="coerce")
    if mask_ms.any():
        parsed.loc[mask_ms] = pd.to_datetime(s.loc[mask_ms], unit="ms", errors="coerce")
    return parsed


def load_and_preprocess(file_path):
    df = pd.read_csv(file_path)
    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    if "close" not in df.columns or "time" not in df.columns:
        raise ValueError(f"{os.path.basename(file_path)} must contain 'close' and 'time' columns")
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df["time"]  = pd.to_numeric(df["time"],  errors="coerce")
    df = df.dropna(subset=["close", "time"]).copy()
    df["parsed_time"] = parse_mixed_time(df["time"], BASE_DATE)
    df = df.dropna(subset=["parsed_time"]).copy()
    df = df.sort_values("parsed_time").reset_index(drop=True)
    df = df.drop_duplicates(subset=["parsed_time"], keep="last")
    df["return_1step"] = df["close"].pct_change()
    df = df.dropna(subset=["return_1step"]).copy()
    return df[["parsed_time", "close", "return_1step"]].reset_index(drop=True)


def split_series(df, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    return (
        df.iloc[:train_end].copy().reset_index(drop=True),
        df.iloc[train_end:val_end].copy().reset_index(drop=True),
        df.iloc[val_end:].copy().reset_index(drop=True),
    )


def fit_arima_safe(series_values, order):
    return ARIMA(series_values, order=order,
                 enforce_stationarity=False, enforce_invertibility=False).fit()


def returns_to_price_path(start_price, forecast_returns):
    prices, prev = [], float(start_price)
    for r in forecast_returns:
        prev = prev * (1.0 + float(r))
        prices.append(prev)
    return np.array(prices)


def calc_metrics(actual, predicted):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2   = r2_score(actual, predicted)
    return mae, rmse, r2


def forecast_is_valid(arr, expected_len):
    arr = np.asarray(arr)
    return len(arr) == expected_len and not np.isnan(arr).any() and not np.isinf(arr).any()


def process_coin(file_name, order, data_dir, output_dir,
                 train_ratio, val_ratio, test_ratio, rebal_interval, base_date):
    """Full walk-forward test for one coin — designed to be called in parallel."""
    file_path = os.path.join(data_dir, file_name)
    try:
        df = load_and_preprocess(file_path)
        n = len(df)
        train_end = int(n * train_ratio)
        val_end   = int(n * (train_ratio + val_ratio))
        train = df.iloc[:train_end].reset_index(drop=True)
        val   = df.iloc[train_end:val_end].reset_index(drop=True)
        test  = df.iloc[val_end:].reset_index(drop=True)

        if len(train) < 30 or len(val) < rebal_interval or len(test) < rebal_interval:
            return file_name, None, "not enough data after split"

        pre_test = pd.concat([train, val], ignore_index=True)
        forecasts, actuals, timestamps = [], [], []

        for idx in range(0, len(test), rebal_interval):
            if idx + rebal_interval > len(test):
                break
            history = pd.concat(
                [pre_test["return_1step"], test["return_1step"].iloc[:idx]],
                ignore_index=True
            )
            last_price    = pre_test["close"].iloc[-1] if idx == 0 else test["close"].iloc[idx - 1]
            actual_return = (float(test["close"].iloc[idx + rebal_interval - 1]) / float(last_price)) - 1.0
            try:
                fitted   = fit_arima_safe(history.to_numpy(dtype=float), order)
                fc       = fitted.forecast(steps=rebal_interval)
                if not forecast_is_valid(fc, rebal_interval):
                    raise ValueError("invalid forecast")
                fp = returns_to_price_path(last_price, np.asarray(fc))
                forecasts.append((fp[-1] / float(last_price)) - 1.0)
                actuals.append(actual_return)
                timestamps.append(test["parsed_time"].iloc[idx])
            except Exception:
                continue

        if len(forecasts) < 5:
            return file_name, None, "too few valid test rebalances"

        test_mae, test_rmse, test_r2 = calc_metrics(actuals, forecasts)
        per_coin_df = pd.DataFrame({
            "timestamp":          timestamps,
            "crypto":             file_name,
            "actual_return_7d":   actuals,
            "forecast_return_7d": forecasts,
        })
        per_coin_df.to_csv(os.path.join(output_dir, f"{file_name}_rebalance_7d_test_forecasts.csv"), index=False)

        result = {
            "crypto":              file_name,
            "order_used":          order,
            "num_test_rebalances": len(forecasts),
            "test_mae":            test_mae,
            "test_rmse":           test_rmse,
            "test_r2":             test_r2,
            "timestamps":          timestamps,
            "forecasts":           forecasts,
            "actuals":             actuals,
        }
        return file_name, result, None

    except Exception as e:
        return file_name, None, str(e)

## Load selected orders

In [ ]:
if not os.path.exists(selected_orders_path):
    raise FileNotFoundError(
        f"Cannot find: {selected_orders_path}\nRun 01_arima_order_selection.ipynb first."
    )

selected_orders_df = pd.read_csv(selected_orders_path)
print(f"Loaded {len(selected_orders_df)} coins from: {selected_orders_path}")
display(selected_orders_df)

## Run walk-forward test (parallelised across coins)

In [ ]:
start_time = time.perf_counter()
print(f"Processing {len(selected_orders_df)} coins in parallel (n_jobs={N_JOBS})...\n")

coin_args = [
    (row["crypto"], ast.literal_eval(row["order_used"]),
     data_dir, output_dir, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, REBAL_INTERVAL, BASE_DATE)
    for _, row in selected_orders_df.iterrows()
]

raw_results = Parallel(n_jobs=N_JOBS)(
    delayed(process_coin)(*args) for args in coin_args
)

summary_rows, all_forecast_rows, all_actual_rows, skipped = [], [], [], []

for file_name, result, error in raw_results:
    if result is None:
        skipped.append((file_name, error))
        print(f"  {file_name}: skipped — {error}")
        continue
    print(f"  {file_name}: order={result['order_used']}  Test RMSE={result['test_rmse']:.8f}")
    summary_rows.append({k: result[k] for k in ["crypto","order_used","num_test_rebalances","test_mae","test_rmse","test_r2"]})
    all_forecast_rows.append(pd.DataFrame({"timestamp": result["timestamps"], "crypto": file_name, "value": result["forecasts"]}))
    all_actual_rows.append(pd.DataFrame({"timestamp": result["timestamps"], "crypto": file_name, "value": result["actuals"]}))

print(f"\nDone. Runtime: {time.perf_counter() - start_time:.2f}s")

## Save outputs

In [ ]:
summary_df = pd.DataFrame(summary_rows)
if len(summary_df) > 0:
    summary_df = summary_df.sort_values("test_rmse")
    summary_df.to_csv(os.path.join(output_dir, "arima_7d_rebalance_test_summary.csv"), index=False)
    print("Test summary:")
    display(summary_df)

if all_forecast_rows and all_actual_rows:
    forecast_matrix = (pd.concat(all_forecast_rows, ignore_index=True)
                       .pivot(index="timestamp", columns="crypto", values="value")
                       .sort_index().sort_index(axis=1))
    actual_matrix   = (pd.concat(all_actual_rows, ignore_index=True)
                       .pivot(index="timestamp", columns="crypto", values="value")
                       .sort_index().sort_index(axis=1))

    common_idx  = forecast_matrix.index.intersection(actual_matrix.index)
    common_cols = forecast_matrix.columns.intersection(actual_matrix.columns)
    forecast_matrix = forecast_matrix.loc[common_idx, common_cols].dropna(how="any")
    actual_matrix   = actual_matrix.loc[common_idx, common_cols]

    for name, mat in [("forecast", forecast_matrix), ("actual", actual_matrix)]:
        mat.to_csv(os.path.join(output_dir, f"ARIMA_test_7d_rebalance_{name}_matrix.csv"))
        np.save(os.path.join(output_dir,    f"ARIMA_test_7d_rebalance_{name}_matrix.npy"), mat.to_numpy())
        print(f"{name.capitalize()} matrix shape: {mat.shape}")

if skipped:
    pd.DataFrame(skipped, columns=["crypto", "reason"]).to_csv(
        os.path.join(output_dir, "arima_7d_rebalance_test_skipped.csv"), index=False)
    print("Skipped:", skipped)